In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# weighting method 비교: CUE(softmax, 학습된 modality 가중치) vs FP-only vs IMG-only vs
# UNIFORM(고정 0.5/0.5) vs Random(Dirichlet(1,1)에서 1회 뽑아 고정한 가중치, 서로 다른 seed로 N회 반복).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# Random 시도 횟수: cue/scripts/run_cue_random_*.sh가 R1..R{NUM_RANDOM_TRIALS} 폴더를 생성했다.
# 필요하면 이 값을 바꿔서(N을 줄여서) 앞쪽 N개 시행만 사용할 수 있다.
NUM_RANDOM_TRIALS = 30

# 고정(단일 결과) 모델 소스. CUE는 표/그래프에서 항상 마지막에 오도록 순서를 맞춘다.
model_sources = {
    'FP': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'FP'),
    'IMG': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'IMG'),
    'UNIFORM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'UNIFORM'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}
# Random 소스: {trial}을 R1..R{NUM_RANDOM_TRIALS}로 치환해 사용
random_template = os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'R{trial}')

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

# 기준선(axhline) 색상: FP/IMG/UNIFORM/CUE. weight_Plot.ipynb와 동일하게
# CUE=06d6a0으로 통일하고(FP/IMG도 동일 매핑으로 맞춤), UNIFORM은 기존 118ab2를 유지한다.
line_colors = {'FP': '#ef476f', 'IMG': '#ffd166', 'CUE': '#06d6a0', 'UNIFORM': '#118ab2'}
# Random swarm 점 색상: 위 4색이 모두 기준선에 쓰이므로 겹치지 않는 중립색을 사용한다.
random_swarm_color = '#495057'


In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


# CUE / FP / IMG / UNIFORM (고정 모델) 로드
raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_fixed = pd.DataFrame(raw_rows)

# Random (R1..R{NUM_RANDOM_TRIALS}) 로드: 시행별로 개별 보관한다 (평균내지 않음 -> p-value 계산에 필요).
raw_rows_random = []
for d_name in datasets:
    for target in targets:
        for trial in range(1, NUM_RANDOM_TRIALS + 1):
            dir_template = random_template.replace('{trial}', str(trial))
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows_random.append({'Dataset': d_name, 'Target': target, 'Trial': trial, **vals})

df_random = pd.DataFrame(raw_rows_random)

print(f"✅ 고정 모델(CUE/FP/IMG/UNIFORM) 데이터 로드 완료: {len(df_fixed)}개")
print(f"✅ Random 시행 데이터 로드 완료: {len(df_random)}개 (기대값: {len(datasets)*len(targets)*NUM_RANDOM_TRIALS})")

✅ 고정 모델(CUE/FP/IMG/UNIFORM) 데이터 로드 완료: 32개
✅ Random 시행 데이터 로드 완료: 240개 (기대값: 240)


In [3]:
def plot_weight_comparison(df_random, df_fixed, dataset, target_name, metric_name, output_dir):
    """Random 시행들의 metric 분포를 swarmplot으로, CUE/FP/IMG/UNIFORM은 기준선(axvline)으로 표시한다.
    가로 방향(값이 x축)으로 표시하며, 축 제목(metric 이름)은 생략한다."""
    df_r = df_random[(df_random['Dataset'] == dataset) & (df_random['Target'] == target_name)]
    if df_r.empty or metric_name not in df_r.columns:
        return

    values = df_r[metric_name].dropna().values
    if len(values) == 0:
        return

    fig, ax = plt.subplots(figsize=(6, 3.5))

    sns.swarmplot(x=values, ax=ax, color=random_swarm_color, size=6, alpha=0.85)

    # FP, IMG, UNIFORM, CUE 순으로 그려 범례도 이 순서로 표시되게 한다 (CUE가 마지막).
    for model_name in ['FP', 'IMG', 'UNIFORM', 'CUE']:
        color = line_colors[model_name]
        row = df_fixed[(df_fixed['Dataset'] == dataset) & (df_fixed['Target'] == target_name) & (df_fixed['Model'] == model_name)]
        if row.empty or metric_name not in row.columns:
            continue
        ref_val = row[metric_name].values[0]
        ax.axvline(ref_val, color=color, linestyle='--', linewidth=2, label=f'{model_name} ({ref_val:.3f})')

    ax.set_title(f"{dataset} / {target_name} / {metric_name}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_yticks([])
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.legend(loc='best', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'weight_comparison_v2', f'{NUM_RANDOM_TRIALS}')

for d_name in datasets:
    for target in targets:
        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            plot_weight_comparison(df_random, df_fixed, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/DIS/PI_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/DIS/PI_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/DIS/Ssel_pcc.png


... (14 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/INH/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/INH/RS(k=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/weight_comparison_v2/30/INH/RS(k=2)_pcc.png
